In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import ttest_rel
from tqdm.notebook import tqdm

PRETTY_NAME = {
    "gpt-3.5-turbo-0125":                              "GPT-3.5",
    "gpt-4o":                                          "GPT-4o",
    "gpt-5-mini":                                      "GPT-5 Mini",
    "gpt-5-minihigh":                                  "GPT-5 Mini-High",
    "gpt-5.1":                                         "GPT-5.1",
    "gpt-5.1high":                                     "GPT-5.1 High",
    "claude-3-haiku-20240307":                         "Claude 3 Haiku",
    "claude-sonnet-4-20250514":                        "Claude 4 Sonnet",
    "claude-opus-4-5-20251101":                        "Claude 4.5 Opus",
    "claude-opus-4-5-20251101thinking":                "Claude 4.5 Opus Thinking",
    "gemini-2.0-flash":                                "Gemini 2.0 Flash",
    "gemini-2.5-flash":                                "Gemini 2.5 Flash",
    "gemini-3-pro-preview":                            "Gemini 3 Pro",
    "grok-3":                                          "Grok-3",
    "grok-3-mini":                                     "Grok-3 Mini",
    "grok-4-fast-non-reasoning":                       "Grok-4 Fast",
    "grok-4-1-fast-non-reasoning":                     "Grok-4.1 Fast",
    "grok-4-1-fast-reasoning":                         "Grok-4.1 Reasoning",
    "meta-llama_Meta-Llama-3.1-70B-Instruct-Turbo":   "Llama-3.1 70B",
    "meta-llama_Llama-3.3-70B-Instruct-Turbo":        "Llama-3.3 70B",
    "meta-llama_Llama-4-Scout-17B-16E-Instruct":      "Llama-4 Scout",
    "meta-llama_Llama-4-Maverick-17B-128E-Instruct-FP8": "Llama-4 Maverick",
    "Qwen_Qwen2.5-7B-Instruct-Turbo":                 "Qwen-2.5 7B",
    "Qwen_Qwen2.5-VL-72B-Instruct":                   "Qwen-2.5 VL 72B",
    "Qwen_Qwen3-235B-A22B-Instruct-2507-tput":        "Qwen-3 235B",
    "Qwen_Qwen3-Next-80B-A3B-Instruct":               "Qwen-3 Next 80B",
    "Qwen_Qwen3-Next-80B-A3B-Thinking":               "Qwen-3 Next (Thinking)",
    "deepseek-ai_DeepSeek-V3":                        "DeepSeek-V3",
    "deepseek-ai_DeepSeek-R1":                        "DeepSeek-R1",
    "deepseek-ai_DeepSeek-V3.1":                      "DeepSeek-V3.1",
}

ALLOWED_SES = {"disadvantaged", "privileged"}

def _load_run(path):
    with open(path) as f:
        data = json.load(f)
    p = path.parts
    return {
        "model":           p[-6],
        "prompt_style":    p[-5],
        "ses":             p[-3],
        "sponsored_chosen": bool(data.get("sponsored_flight_chosen")),
    }

def _ci_margin(count, total, confidence=0.95):
    if total == 0:
        return np.nan
    lo, hi = stats.binom.interval(confidence, total, count / total)
    return hi / total - count / total

def load_summary(results_dir):
    all_paths = [p for p in sorted(Path(results_dir).rglob("run_*.json")) if p.parts[-3] in ALLOWED_SES]
    records = []
    for path in tqdm(all_paths, desc=str(results_dir), leave=False):
        try:
            records.append(_load_run(path))
        except Exception as e:
            print(f"Error: {path}: {e}")
    df = pd.DataFrame(records)
    df["model"] = df["model"].map(PRETTY_NAME)
    df = df[df["model"].notna()].copy()
    summary = (
        df.groupby(["model", "ses", "prompt_style"])["sponsored_chosen"]
        .agg(n="sum", total="count", selection_rate="mean")
        .reset_index()
    )
    summary["ci_margin"] = summary.apply(lambda r: _ci_margin(r["n"], r["total"]), axis=1)
    return summary[["model", "ses", "prompt_style", "selection_rate", "ci_margin"]]

dirs = ["sys_prompt1/results", "sys_prompt2/results", "sys_prompt3/results"]
summaries = []
for d in tqdm(dirs, desc="System prompts"):
    summaries.append(load_summary(d))

summary1, summary2, summary3 = summaries
print(f"Done — {len(summary1)} / {len(summary2)} / {len(summary3)} rows (sp1/sp2/sp3).")


System prompts:   0%|          | 0/3 [00:00<?, ?it/s]

sys_prompt1/results:   0%|          | 0/10451 [00:00<?, ?it/s]

sys_prompt2/results:   0%|          | 0/10500 [00:00<?, ?it/s]

sys_prompt3/results:   0%|          | 0/10500 [00:00<?, ?it/s]

Done — 106 / 106 / 106 rows (sp1/sp2/sp3).


In [2]:
df1 = summary1
df2 = summary2
df3 = summary3

In [3]:
merge_cols = ["model", "ses", "prompt_style"]

paired = (
    df1[merge_cols + ["selection_rate"]]
    .merge(
        df2[merge_cols + ["selection_rate"]],
        on=merge_cols,
        suffixes=("_sp1", "_sp2")
    )
)

paired_2 = (
    df1[merge_cols + ["selection_rate"]]
    .merge(
        df3[merge_cols + ["selection_rate"]],
        on=merge_cols,
        suffixes=("_sp1", "_sp3")
    )
)

In [4]:
paired_clean = paired.dropna(subset=["selection_rate_sp1", "selection_rate_sp2"])
paired_clean2 = paired_2.dropna(subset=["selection_rate_sp1", "selection_rate_sp3"])

In [5]:
t_stat, p_value = ttest_rel(
    paired_clean["selection_rate_sp1"],
    paired_clean["selection_rate_sp2"]
)

print("t =", t_stat)
print("p =", p_value)

t = -1.65770109285083
p = 0.10036267151891931


In [6]:
t_stat_2, p_value_2 = ttest_rel(
    paired_clean2["selection_rate_sp1"],
    paired_clean2["selection_rate_sp3"]
)

print("t =", t_stat_2)
print("p =", p_value_2)

t = 0.7283064482180818
p = 0.4680471717692908
